Equipo Integrantes:
- Carla Sophia Rodríguez Dander - A01781793
- Isaac Husny - A01027140
- Jesus Rodríguez - A01025112


In [ ]:
! pip install category_encoders

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 1.6 MB/s eta 0:00:00


In [ ]:
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import joblib
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

# ***1.***

#Carga de Archivos Demográfico e Histórico

In [ ]:
#importamos los archivos desde Google Drive
drive.mount('/content/drive')
path='/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/'
Historico=pd.read_csv(path+'RenunciasHistorico.csv')
Demo=pd.read_csv(path+'RenunciasDemo.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<ipython-input-43-483cd7b7174c>:3: DtypeWarning: Columns (5,11) have mixed types. Specify dtype option on import or set low_memory=False.
  Historico=pd.read_csv(path+'RenunciasHistorico.csv')


#Crear Funciones

In [ ]:
#Funciones
#1. Remplazo de valores str a valores int.
def stringaint(df, columna_origen, columna_destino):
  #Diccionario que mapea cada valor unico desde la columna de origen a su identificador int correspondiente de columna destino
  DiccMap = df.drop_duplicates().set_index(columna_origen)[columna_destino].to_dict()
  #Filtro en el diccionario de mapeo para retener solo valores enteros
  DiccMap={k: v for k, v in DiccMap.items() if isinstance(v, int)}
  # Reemplazamos valores string en la columna de clave por valores int
  df[columna_destino] = df.apply(lambda row: DiccMap.get(row[columna_origen], row[columna_destino]), axis=1)
  return df

#2. Identificación de valores duplicados.
def revisarDuplicados(df):
  #Localizamos todas las filas en donde los valores son los mismos.
  mask= (df.T == df.iloc[:, 0]).all()
  #Obtiene los índices de las filas con valores uniformes
  uniform_row_indices= df[mask].index.tolist()

  return len(uniform_row_indices), uniform_row_indices

#3. Frecuencia de ocurrencias
def crear_csv_ref(df, output_file_path, column1, column2):
    # Llenar valores NaN en las columnas con "Faltante"
    df[[column1, column2]] = df[[column1, column2]].fillna("Faltante")
    # Calcular la frecuencia de ocurrencias para cada par único de valores de columna1 y columna2
    df_counts = df.groupby([column1, column2]).size().reset_index(name='Conteo')
    # Reemplazar "Faltante" de vuelta a NaN si es necesario
    df_counts.replace("Faltante", np.nan, inplace=True)
    # Guardar el DataFrame en el nuevo archivo CSV
    df_counts.to_csv(output_file_path,encoding='utf-8', index=False)

#4. Eliminación de columnas especificas
def eliminar_cols_desc(df, columnas_a_eliminar):
  #Elimina las columnas especificadas
  if set(columnas_a_eliminar).issubset(df.columns):
    df.drop(columns=columnas_a_eliminar, inplace=True)
    #Guardamos el DataFrame modificado en el mismo archivo CSV
  else:
    print(f"¡Algunas columnas de {columnas_a_eliminar} no se encontraron en el CSV!")

#5. llenado de valores vacios
def fill_empty_values(df,col_name, text1):
  #Usamos fillna para llenar los valores vacíos en la segunda columna
  df[col_name].fillna(text1, inplace=True)
  #Guardamos en el mismo archivo
  print(f"Valores vacíos en la columna '{col_name}' han sido llenados con '{text1}'.")

#6. Convertir valores a mayuscula
def convert_column_to_uppercase(df, col_name):
    #Pasamos la columna especificada a mayúsculas
    df[col_name] = df[col_name].str.upper()

#7. Conversión y manejo de fechas, separación en columnas.
def split_date_column(df, date_column, year_col="year", month_col="month", day_col="day"):
    # Convert to string first to handle '9999-12-31' and other out-of-bounds dates
    df[date_column] = df[date_column].astype(str)
    # Replace out-of-bounds dates with a placeholder or NaN
    df[date_column] = df[date_column].replace('9999-12-31', pd.NA)
    # Now convert to datetime format
    df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
    # Determine the location for the new columns
    col_loc = df.columns.get_loc(date_column)

    # Extract and cast columns to integer (if NaN values exist, they'll stay as floats)
    df.insert(col_loc + 1, year_col, df[date_column].dt.year.fillna(0).astype(int))
    df.insert(col_loc + 2, month_col, df[date_column].dt.month.fillna(0).astype(int))
    df.insert(col_loc + 3, day_col, df[date_column].dt.day.fillna(0).astype(int))

    return df

#8.Resume la distribución y el porcentaje de aparición de categorías en una columna del DataFrame.
def cuenta_por_cat(df,col):
  year_counts = df.groupby(col).size()
  total_rows = len(df)
  year_percentages = year_counts / total_rows * 100
  result = pd.DataFrame({
       'Conteo': year_counts,
       'Porcentaje': year_percentages
  })
  print(result)

#9. Eliminación de las filas NaT
def remove_nat_rows(df, date_column):
    return df[df[date_column].notna()]
def remove_rows_after_year(df, date_column, year):
    return df[df[date_column].dt.year <= year]

#Llamar Funciones

###Visualización previa de las columnas sin modificar

In [ ]:
#Visualisamos los dos df para validar que esten cargados correctamente
display(Historico.head(n=2))
display(Demo.head(n=2))


,Nº pers.,Hasta,Desde,Soc.,Desc Soc,DivP,Desc Div Per,GrPer,Desc Gpo Personal,ÁPers,Desc APers,Clave de organización,Ubicación,Un.org.,Desc Un Org,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,SELI,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,12/31/1999,1/1/1999,140,Servicios Liverpool S.A.,SELI,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Ubicación,Descubica,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,Gpo Per,Desc Gpo Pers,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000225,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,31,11500,234,986,Suburbia Los Cabos Patio,2022-11-03,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000232,Supervisor Cajas


Agregamos una columna en donde se binarizo la columna genero en donde hombre=0 y mujer=1

In [ ]:
"""
Demo['Genero_Binario'] = Demo['Genero'].apply(genero_a_binario)
# Reorganiza las columnas para colocar 'Genero_Binario' al lado de 'Genero'
Demo = Demo[['Genero', 'Genero_Binario'] + [col for col in Demo.columns if col != 'Genero' and col != 'Genero_Binario']]
display(Demo.head(n=2))
"""

"\nDemo['Genero_Binario'] = Demo['Genero'].apply(genero_a_binario)\n# Reorganiza las columnas para colocar 'Genero_Binario' al lado de 'Genero'\nDemo = Demo[['Genero', 'Genero_Binario'] + [col for col in Demo.columns if col != 'Genero' and col != 'Genero_Binario']]\ndisplay(Demo.head(n=2))\n"

Cambiamos los nombres de etiquetas para que todo tuviera el mismo codigo y estos fueran más claros a la hora de leer la info.

###Cambio de Nombre de las columnas

In [ ]:
#Cambio de nombre de columnas para identificar de forma mas facil

Historico= Historico.rename(columns={'DivP':'Clave Ubicación',
                                     'Desc Div Per':'Locación',
                                     'Clave de organización': 'ID Depa',
                                     'Ubicación':'Departamento',
                                     ' Un.org.':'ID Unidad',
                                     'Desc Un Org':'Unidad',
                                     'ÁPers': 'ID Área Personal',
                                     'Desc APers':'Área Personal',
                                     'GrPer': 'ID GroPer',
                                     'Desc Gpo Personal':'GroPer'
                                     })

Demo= Demo.rename(columns={'Ubicación':'Clave Ubicación',
                           'Descubica':'Locación',
                           'Gpo Per': 'ID GroPer',
                           'Desc Gpo Pers':'GroPer'})

display(Historico.head(n=1))
display(Demo.head(n=1))

,Nº pers.,Hasta,Desde,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,SELI,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000225,Jefe Prevencion Perdidas


###De str a int
En el siguiente cuadro de codigo, creamos una función que nos permitia primero identificar todas las combiaciones que usaban cadenas, y enseguida cambia todos estos elementos que estaban en string a ints.

In [ ]:
#En el siguiente cuadro de codigo,
#creamos una función que nos permitia primero identificar todas las combiaciones que usaban cadenas,
#y enseguida cambia todos estos elementos que estaban en string a ints.
stringaint(Historico, "Locación", "Clave Ubicación" )
stringaint(Historico, "Departamento", "ID Depa" )
stringaint(Historico, "Unidad", "ID Unidad" )
stringaint(Historico, "Desc Función", " Función" )

stringaint(Demo, "Locación", "Clave Ubicación" )
stringaint(Demo, "Desc Fun", "Función" )

display(Historico.head(n=2))
display(Demo.head(n=2))

,Nº pers.,Hasta,Desde,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,12/31/1999,1/1/1999,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,Hombre,0,2022-08-24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000086,Jefe Prevencion Perdidas
1,70636679,1990-12-23,Mujer,0,2022-08-31,31,11500,234,986,Suburbia Los Cabos Patio,2022-11-03,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000099,Supervisor Cajas


###Clave de CONSEJO y Sin Departamento
Clave de CONSEJO
>Detectamos que la columna ID Depa tiene varias celdas sin valor y la descripcion de estas siempre es CONSEJO. Por ende vamos a agregar un 0 como clave de CONSEJO

Detectamos que algunas claves de departamentos no tienen un departamento asignado, por lo mismo estaremos asignando "Sin departamento" a la descripcion de esta en el archivo Departamentos.csv

>El código llena los valores vacíos en la segunda columna de un archivo CSV con el texto "Sin Departamento", utilizando la función "fillna" para reemplazar los valores vacíos en la segunda columna (especificada por "col_name") con el texto proporcionado ("text1") y al finalizar, se guarda el DataFrame modificado en el mismo archivo CSV y se muestra el DataFrame actualizado en la última línea del código.



In [ ]:
Historico['ID Depa'].fillna(0, inplace=True)
display(Historico.head(n=2))
display(Demo.head(n=0))

,Nº pers.,Hasta,Desde,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,12/31/1998,12/1/1974,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,12/31/1999,1/1/1999,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


,Nº pers.,Fecha nacimiento,Genero,No Hijos,Fecha ingreso,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun


In [ ]:
#Identificamos que la columna tiene algunos celdas vacias, por lo cual las completamos con 2"SIN DEPTO"
fill_empty_values(Historico, 'Departamento', 'SIN DEPTO')

Valores vacíos en la columna 'Departamento' han sido llenados con 'SIN DEPTO'.


###Valores duplicados
En el siguiente codigo revisamos cuantas filas tienen todos los valores duplicados, para en su caso valorarlas y eliminarlas si es necesario.

>El cod. funciona para identificar y contar las filas en un archivo CSV en las cuales todos los valores son iguales en todas las columnas. Luego, devuelve el número de estas filas y una lista de sus índices.

Nuestro resultado fue 0 por lo que no hubo necesidad de eliminar nada.

In [ ]:
#En el siguiente codigo revisamos cuantas filas tienen todos los valores duplicados, para en su caso valorarlas y eliminarlas si es necesario.
numero_filas_uniformes, indices_filas_uniformes = revisarDuplicados(Historico)

print(f"Número de filas con valores uniformes: {numero_filas_uniformes}")
print("Índices de filas con valores uniformes:")

for indice in indices_filas_uniformes:
    print(indice)

Número de filas con valores uniformes: 0
Índices de filas con valores uniformes:


###Formato de Fecha
Se transformo la columna **Hasta** al siguiente formato: MM-DD-AAAA  y se agregaron como columnas separadas enseguida de la columna Hasta con la información del día, mes, año, hora y minuto. La problematica de este punto es que la fecha contenia segundos y valores NAN

> Utilizamos *pd.to_datetime* con *errors='coerce'* para manejar las fechas y aplicar el formato deseado. Extraímos las partes de la fecha y la hora utilizando expresiones regulares y las convertimos a enteros para las columnas separadas.

In [ ]:
#separamos columnas de fecha en dia, mes y año
split_date_column(Historico, 'Hasta', 'HAño', 'HMes', 'HDia')
split_date_column(Historico, 'Desde', 'DAño', 'DMes', 'DDia')
split_date_column(Demo, 'Fecha nacimiento', 'NAño', 'NMes', 'NDia')
split_date_column(Demo, 'Fecha ingreso', 'IAño', 'IMes', 'IDia')
split_date_column(Demo, 'Fecha Salida', 'SAño', 'SMes', 'SDia')

,Nº pers.,Fecha nacimiento,NAño,NMes,NDia,Genero,No Hijos,Fecha ingreso,IAño,IMes,IDia,Edad ingreso,CP Vivienda,CP Trabajo,Clave Ubicación,Locación,Fecha Salida,SAño,SMes,SDia,Antigüedad,Edad salida,Año salida,Desc Medida,ID GroPer,GroPer,Ultima Evaluación,Función,Desc Fun
0,70634620,1984-02-28,1984,2,28,Hombre,0,2022-08-24,2022,8,24,38,2770,234,986,Suburbia Los Cabos Patio,2022-10-08,2022,10,8,0,38,2022,Baja Suburbia,O,SBB Planta,3.0,70000086,Jefe Prevencion Perdidas
1,70636679,1990-12-23,1990,12,23,Mujer,0,2022-08-31,2022,8,31,31,11500,234,986,Suburbia Los Cabos Patio,2022-11-03,2022,11,3,0,31,2022,Baja Suburbia,O,SBB Planta,3.0,70000099,Supervisor Cajas
2,70642442,1981-06-20,1981,6,20,Mujer,0,2022-09-28,2022,9,28,41,23410,234,986,Suburbia Los Cabos Patio,2022-11-22,2022,11,22,0,41,2022,Baja Suburbia,O,SBB Planta,3.0,70000103,Auxiliar Cajero
3,70639827,1993-05-02,1993,5,2,Hombre,0,2022-09-14,2022,9,14,29,23428,234,986,Suburbia Los Cabos Patio,2022-10-26,2022,10,26,0,29,2022,Baja Suburbia,O,SBB Planta,3.0,70000098,Subjefe Proteccion
4,13111365,1993-08-29,1993,8,29,Mujer,0,2014-11-13,2014,11,13,21,23450,234,986,Suburbia Los Cabos Patio,2022-10-14,2022,10,14,7,29,2022,Baja Suburbia,O,SBB Planta,3.0,0,Sin Función
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117155,15178424,1994-10-04,1994,10,4,Hombre,0,2019-10-28,2019,10,28,25,Sin Dato,82103,429,Sfera Mazatlán,2019-11-30,2019,11,30,0,25,2019,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera
117156,15200769,1997-09-29,1997,9,29,Mujer,0,2019-11-13,2019,11,13,22,Sin Dato,97133,409,Sfera Mérida,2020-02-01,2020,2,1,0,22,2020,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera
117157,15273135,1999-09-18,1999,9,18,Hombre,0,2020-02-17,2020,2,17,20,Sin Dato,82103,429,Sfera Mazatlán,2020-05-20,2020,5,20,0,20,2020,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera
117158,15280203,1990-12-09,1990,12,9,Mujer,0,2020-02-26,2020,2,26,29,Sin Dato,82103,429,Sfera Mazatlán,2020-05-13,2020,5,13,0,29,2020,Baja,D,Vía Planta No Sind.,3.0,823,Vendedor Boutique Sfera


In [ ]:
display(Historico.head(n=2))

,Nº pers.,Hasta,HAño,HMes,HDia,Desde,DAño,DMes,DDia,Soc.,Desc Soc,Clave Ubicación,Locación,ID GroPer,GroPer,ID Área Personal,Área Personal,ID Depa,Departamento,ID Unidad,Unidad,Función,Desc Función
0,3369,1998-12-31,1998,12,31,1974-12-01,1974,12,1,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,0,Sin Un Org,0,Sin Función
1,3369,1999-12-31,1999,12,31,1999-01-01,1999,1,1,140,Servicios Liverpool S.A.,0,Servicios Liverpool,B,Planta No Sind.,1E,Ejecutivo/ Coord.,540,JUNIORS,40,Compras Juniors,83,Comprador Sr B


###Orden

In [ ]:
#Reacomodamos el orden de las columnas
column_orderD= ['Nº pers.',	'Fecha nacimiento', 'NAño', 'NMes', 'NDia', 'Genero','No Hijos', 'Fecha ingreso','IAño',	'IMes',
                'Edad ingreso',	'Fecha Salida',	'SAño',	'SMes', 'Antigüedad','Edad salida',	'Año salida',
                'Desc Medida',	'GroPer',	'Ultima Evaluación', 'CP Vivienda','CP Trabajo','Locación','Desc Fun']

Demo=Demo[column_orderD]
display(Demo.head(n=2))

,Nº pers.,Fecha nacimiento,NAño,NMes,NDia,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,SMes,Antigüedad,Edad salida,Año salida,Desc Medida,GroPer,Ultima Evaluación,CP Vivienda,CP Trabajo,Locación,Desc Fun
0,70634620,1984-02-28,1984,2,28,Hombre,0,2022-08-24,2022,8,38,2022-10-08,2022,10,0,38,2022,Baja Suburbia,SBB Planta,3.0,2770,234,Suburbia Los Cabos Patio,Jefe Prevencion Perdidas
1,70636679,1990-12-23,1990,12,23,Mujer,0,2022-08-31,2022,8,31,2022-11-03,2022,11,0,31,2022,Baja Suburbia,SBB Planta,3.0,11500,234,Suburbia Los Cabos Patio,Supervisor Cajas


In [ ]:
#Reacomodamos el orden de las columnas

column_orderH= ['Nº pers.', 'Hasta', 'HAño', 'HMes', 'Desde', 'DAño', 'DMes',
                'Desc Soc','Locación', 'GroPer','Área Personal',
                'Departamento', 'Unidad', 'Desc Función']
Historico=Historico[column_orderH]
display(Historico.head(n=2))

,Nº pers.,Hasta,HAño,HMes,Desde,DAño,DMes,Desc Soc,Locación,GroPer,Área Personal,Departamento,Unidad,Desc Función
0,3369,1998-12-31,1998,12,1974-12-01,1974,12,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Sin Un Org,Sin Función
1,3369,1999-12-31,1999,12,1999-01-01,1999,1,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B


###Eliminar Filas y columnas NaN y Valores naT

Eliminar columnas donde todos los valores sean nan, paso lo mismo no habia columnas que cumplieran con el requsito, por lo que no se elimino nada

In [ ]:
#Quitamos los valores Nan y revisamos cuantas filas hay despues de quitarlo en cada df
print("# Columnas antes en Doc Historico:",len(Historico.columns))
Historico.dropna(how='all', axis=1, inplace=True)
#Numero de filas en el dataframe después de la operación
print("# Columnas despues en Doc Historico:",len(Historico.columns))

print("# Columnas antes en Doc Demo:",len(Demo.columns))
Demo.dropna(how='all', axis=1, inplace=True)
#Numero de filas en el dataframe después de la operación
print("# Columnas despues en Doc Demo:",len(Demo.columns))

# Columnas antes en Doc Historico: 14
# Columnas despues en Doc Historico: 14
# Columnas antes en Doc Demo: 24
# Columnas despues en Doc Demo: 24


<ipython-input-56-67e706d15fe5>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Demo.dropna(how='all', axis=1, inplace=True)


###Eliminar columnas del Dataframe
Una vez que ternimanos de realizar la limpieza de los datos podemos crear archivos csv con las diferentes claves de cada concepto, sus descripciones y el conteo de la veces que cada una aparece despues de la limpieza. Despues de esto eliminamos las columnas que resguardamos en los nuevos archivos csv para tener una mejor organizacion en el dataframe.

In [ ]:
#Guardamos las decripciones y claves de columnas en archivos separados y eliminamos las columnas de descripcion para reducir numero de columnas
eliminar_cols_desc(Historico, ["Soc.", "ID GroPer","ID Área Personal", "ID Depa", "Unidad"])
eliminar_cols_desc(Demo,["GroPer"])
display(Historico.head(n=4))
display(Demo.head(n=4))

¡Algunas columnas de ['Soc.', 'ID GroPer', 'ID Área Personal', 'ID Depa', 'Unidad'] no se encontraron en el CSV!


<ipython-input-44-ec9380ec446c>:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=columnas_a_eliminar, inplace=True)


,Nº pers.,Hasta,HAño,HMes,Desde,DAño,DMes,Desc Soc,Locación,GroPer,Área Personal,Departamento,Unidad,Desc Función
0,3369,1998-12-31,1998,12,1974-12-01,1974,12,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Sin Un Org,Sin Función
1,3369,1999-12-31,1999,12,1999-01-01,1999,1,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B
2,3369,2004-01-31,2004,1,2000-01-01,2000,1,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr B
3,3369,2005-11-30,2005,11,2004-02-01,2004,2,Servicios Liverpool S.A.,Servicios Liverpool,Planta No Sind.,Ejecutivo/ Coord.,JUNIORS,Compras Juniors,Comprador Sr A


,Nº pers.,Fecha nacimiento,NAño,NMes,NDia,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,SMes,Antigüedad,Edad salida,Año salida,Desc Medida,Ultima Evaluación,CP Vivienda,CP Trabajo,Locación,Desc Fun
0,70634620,1984-02-28,1984,2,28,Hombre,0,2022-08-24,2022,8,38,2022-10-08,2022,10,0,38,2022,Baja Suburbia,3.0,2770,234,Suburbia Los Cabos Patio,Jefe Prevencion Perdidas
1,70636679,1990-12-23,1990,12,23,Mujer,0,2022-08-31,2022,8,31,2022-11-03,2022,11,0,31,2022,Baja Suburbia,3.0,11500,234,Suburbia Los Cabos Patio,Supervisor Cajas
2,70642442,1981-06-20,1981,6,20,Mujer,0,2022-09-28,2022,9,41,2022-11-22,2022,11,0,41,2022,Baja Suburbia,3.0,23410,234,Suburbia Los Cabos Patio,Auxiliar Cajero
3,70639827,1993-05-02,1993,5,2,Hombre,0,2022-09-14,2022,9,29,2022-10-26,2022,10,0,29,2022,Baja Suburbia,3.0,23428,234,Suburbia Los Cabos Patio,Subjefe Proteccion


#

#Eliminación de Eventuales, Outliers y Merge

###Merge entre Histórico y Demo

In [ ]:
Historico = Historico.rename(columns={'Hasta': 'Fecha Salida'})
# Ordena el DataFrame por Nº pers. y Fecha de egreso de forma descendente
df_sorted = Historico.sort_values(by=['Nº pers.', 'Fecha Salida'], ascending=[True, False])

# Selecciona la primera fila para cada Nº pers., que ahora será el último puesto
ultimo_puesto = df_sorted.drop_duplicates(subset='Nº pers.', keep='first')    #elimina los valores duplicados en ID

merged_df = pd.merge(Demo, ultimo_puesto[['Nº pers.', 'GroPer', 'Área Personal', 'Departamento', 'Unidad', 'Desc Soc']],    #mezclamos ambos dataframes (en base al ID del empleado)
                     on=['Nº pers.'], how='left')

len(merged_df)

117160

In [ ]:
# Cuenta los valores nulos en la columna 'Nombre_Columna'
valores_nulos = merged_df['Departamento'].isna().sum()

# Imprime la cantidad de valores nulos
print('Cantidad de valores nulos en la columna:', valores_nulos)    #nos aseguramos de que no haya valores nulos tras hacer el merge

Cantidad de valores nulos en la columna: 0


In [ ]:
# Drop rows with missing values in the specified column
merged_df.dropna(subset=['Departamento'], inplace=True)
merged_df.dropna(subset=['GroPer'], inplace=True)
merged_df.dropna(subset=['Área Personal'], inplace=True)
merged_df.dropna(subset=['Unidad'], inplace=True)
merged_df.dropna(subset=['Desc Soc'], inplace=True)

###Eliminación String "Sin Función" en Columna 'Desc Fun'

In [ ]:
# Contar el número de filas antes de la eliminación en Demo
num_filas_demo_original = len(merged_df)

# Filtrar filas en las que 'Desc Fun' no contiene 'Sin Función' en el DataFrame Demo
merged_df = merged_df[~merged_df['Desc Fun'].str.contains('Sin Función', case=False, na=False)]

# Contar el número de filas después de la eliminación en Demo
num_filas_demo_filtrado = len(merged_df)

# Calcular el número de filas eliminadas en Demo
filas_eliminadas_demo = num_filas_demo_original - num_filas_demo_filtrado

# Mostrar el número de filas eliminadas en Demo
print(f'Se han eliminado {filas_eliminadas_demo} filas del DataFrame merged_df.')

Se han eliminado 1443 filas del DataFrame merged_df.


In [ ]:
# Contar el número de filas antes de la eliminación en Historico
num_filas_historico_original = len(Historico)

# Filtrar filas en las que 'Desc Fun' no contiene 'Sin Función' en el DataFrame Historico
Historico = Historico[~Historico['Desc Función'].str.contains('Sin Función', case=False, na=False)]

# Contar el número de filas después de la eliminación en Historico
num_filas_historico_filtrado = len(Historico)

# Calcular el número de filas eliminadas en Historico
filas_eliminadas_historico = num_filas_historico_original - num_filas_historico_filtrado

# Mostrar el número de filas eliminadas en Historico
print(f'Se han eliminado {filas_eliminadas_historico} filas del DataFrame Historico.')


Se han eliminado 8169 filas del DataFrame Historico.


###Eliminar a trabajadores con menos de 4 meses

In [ ]:
# Calcular la duración en días en el DataFrame Demo
merged_df['Duración Días'] = (pd.to_datetime(merged_df['Fecha Salida']) - pd.to_datetime(merged_df['Fecha ingreso'])).dt.days

# Calcular la duración en días en el DataFrame Historico
Historico['Duración Días'] = (pd.to_datetime(Historico['Fecha Salida']) - pd.to_datetime(Historico['Desde'])).dt.days

In [ ]:
# Contar el número de filas antes de la eliminación en Demo
num_filas_demo_original = len(merged_df)

# Contar el número de filas antes de la eliminación en Historico
num_filas_historico_original = len(Historico)

print(num_filas_demo_original)
print(num_filas_historico_original)

115717
551961


In [ ]:
# Filtrar las filas en Demo con una duración mayor a 121 días
merged_df = merged_df[merged_df['Duración Días'] > 121]
# Filtrar las filas en Historico con una duración mayor a 121 días
Historico = Historico[Historico['Duración Días'] > 121]

# Calcular el número de filas eliminadas en Demo
filas_eliminadas_demo = num_filas_demo_original - len(merged_df)
# Calcular el número de filas eliminadas en Historico
filas_eliminadas_historico = num_filas_historico_original - len(Historico)


# Mostrar el número de filas eliminadas en Demo
print(f'Se han eliminado {filas_eliminadas_demo} filas del DataFrame Demo.')
# Mostrar el número de filas eliminadas en Historico
print(f'Se han eliminado {filas_eliminadas_historico} filas del DataFrame Historico.')

Se han eliminado 53825 filas del DataFrame Demo.
Se han eliminado 393192 filas del DataFrame Historico.


In [ ]:
Demo = merged_df

###Crear nuevos rangos en la columna target 'Antigüedad'

Aquí buscamos crear grupos que contengan alrededor de la misma cantidad de empleados para poder mejorar la precisión del modelo

In [ ]:
#Grupo < 1 Años
column_name = 'Antigüedad'
column_counts = Demo[column_name].value_counts()
column_counts

0     18770
1     12327
2      8516
3      6028
4      4202
5      2573
6      1732
7      1330
8       964
9       721
10      663
11      534
12      460
13      410
15      362
14      350
16      241
18      219
17      213
19      179
20      157
21      142
22       97
23       66
24       54
25       53
26       49
27       46
28       44
31       44
29       44
32       40
30       34
33       33
35       32
34       27
38       26
40       25
37       22
39       22
36       19
41       12
42        5
44        3
43        1
45        1
Name: Antigüedad, dtype: int64

In [ ]:
#Grupo 1-5 Años
desired_range = column_counts.index[1:6]

# Step 3: Sum the values within the desired range
sum_values = column_counts.loc[desired_range].sum()

# Display the result
print("Sum of values from the second index to the third:", sum_values)

Sum of values from the second index to the third: 33646


In [ ]:
#Grupo 5+ Años
desired_range = column_counts.index[6:]

# Step 3: Sum the values within the desired range
sum_values = column_counts.loc[desired_range].sum()

# Display the result
print("Sum of values from the third index to the tenth:", sum_values)

Sum of values from the third index to the tenth: 9476


In [ ]:
# def assign_value(row):
#     if row['Antigüedad'] == 0:
#         return 0
#     elif row['Antigüedad'] == 1 or row['Antigüedad'] == 2:
#         return 1
#     elif 3 <= row['Antigüedad'] <= 10:
#         return 2
#     else:
#         return 3

# Demo['Antigüedad_Grupos'] = Demo.apply(assign_value, axis=1)

In [ ]:
def assign_value(row):
    if row['Antigüedad'] == 0:
        return 0
    elif row['Antigüedad'] == 1 or row['Antigüedad'] == 2 or row['Antigüedad'] == 3 or row['Antigüedad'] == 4 or row['Antigüedad'] == 5:
        return 1
    else:
        return 2

Demo['Antigüedad_Grupos'] = Demo.apply(assign_value, axis=1)

###Eliminación de Outliers en Columnas Numéricas




In [ ]:
#Columna 'No Hijos'-
column_name = 'No Hijos'
column_counts = Demo[column_name].value_counts()
column_counts

0    42308
1     8713
2     6803
3     3286
4      625
5      123
6       27
7        7
Name: No Hijos, dtype: int64

In [ ]:
# Step 1: Count the occurrences of each unique string value
#Contamos las veces que aparece cada string
value_counts = Demo['No Hijos'].value_counts()

# Step 2: Filter string values with a count of less than 100
#
strings_to_drop = value_counts[value_counts < 200].index

# Step 3: Use boolean indexing to drop rows with these string values
Demo = Demo[~Demo['No Hijos'].isin(strings_to_drop)]

In [ ]:
#Columna 'Edad ingreso'
column_name = 'Edad ingreso'
column_counts = Demo[column_name].value_counts()
column_counts

19    4907
18    4701
20    4489
21    4220
22    4208
23    4056
24    3705
25    3318
26    2916
27    2532
28    2157
29    1876
30    1646
31    1408
32    1299
33    1187
34    1051
35     971
37     890
36     886
38     848
39     790
40     773
41     689
42     666
45     587
44     587
43     578
47     501
46     483
48     388
49     379
50     295
51     270
52     214
53     211
54     203
55     197
56     168
57     111
58     103
17      91
59      59
60      40
61      29
62      18
63      15
67       4
65       4
66       3
68       3
64       3
70       1
74       1
Name: Edad ingreso, dtype: int64

In [ ]:
# Calculate value counts for the specified column
value_counts = Demo['Edad ingreso'].value_counts()

# Identify values with counts greater than or equal to 100
values_to_keep = value_counts[value_counts >= 100].index

# Filter rows where the values in the specified column are in the values_to_keep list
Demo = Demo[Demo[column_name].isin(values_to_keep)]

In [ ]:
pd.options.display.max_columns = None
Demo

,Nº pers.,Fecha nacimiento,NAño,NMes,NDia,Genero,No Hijos,Fecha ingreso,IAño,IMes,Edad ingreso,Fecha Salida,SAño,SMes,Antigüedad,Edad salida,Año salida,Desc Medida,Ultima Evaluación,CP Vivienda,CP Trabajo,Locación,Desc Fun,GroPer,Área Personal,Departamento,Unidad,Desc Soc,Duración Días,Antigüedad_Grupos
19,13307867,1997-03-13,1997,3,13,Mujer,0,2015-06-26,2015,6,18,2022-12-14,2022,12,7,25,2022,Baja Suburbia,3.16,23562,234,Suburbia Los Cabos Patio,Cajero,SBB Planta,SBB Operación,CAJAS,Jefatura Cajas SBB Patio,Suburbia S. de R.L. de CV,2728,2
20,14877876,1991-05-14,1991,5,14,Hombre,0,2019-02-01,2019,2,27,2021-10-16,2021,10,2,30,2021,Baja,3.00,23400,234,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,Ventas Boutique Los Cabos,"Distribuidora Liverpool,",988,1
21,15228049,1992-03-17,1992,3,17,Mujer,2,2019-12-01,2019,12,27,2022-12-07,2022,12,3,30,2022,Baja,3.00,23450,234,Boutique Los Cabos,Consejero de Belleza,Vía Planta No Sind.,Personal General,FRAGANCIAS,Boutique Cabo San Lucas,"Distribuidora Liverpool,",1102,1
22,12379884,1970-03-31,1970,3,31,Mujer,0,2012-09-20,2012,9,42,2020-02-01,2020,2,7,49,2020,Baja,2.96,23456,234,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,Ventas Boutique Los Cabos,Operadora Comercial Liver,2690,2
23,14559473,1965-01-10,1965,1,10,Mujer,0,2018-05-15,2018,5,53,2022-11-04,2022,11,4,57,2022,Baja,1.88,23456,234,Boutique Los Cabos,Consejero de Belleza,Planta No Sind.,Personal General,FRAGANCIAS,Boutique Cabo San Lucas,"Distribuidora Liverpool,",1634,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117144,14503279,1996-07-18,1996,7,18,Hombre,0,2018-04-04,2018,4,21,2019-04-04,2019,4,1,22,2019,Baja,2.08,Sin Dato,5109,Sfera Tlaquepaque,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Tlaquepaq,Operadora Sfera México SA,365,1
117145,14583940,1996-01-01,1996,1,1,Mujer,0,2018-06-06,2018,6,22,2019-01-29,2019,1,0,23,2019,Baja,2.08,Sin Dato,5348,Sfera Playa del Carmen,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Playa del,Operadora Sfera México SA,237,0
117149,14864493,1990-08-30,1990,8,30,Hombre,0,2019-01-18,2019,1,28,2020-01-14,2020,1,0,29,2020,Baja,3.00,Sin Dato,5348,Sfera Guadalajara Gran Plaza,Vendedor Boutique Sfera,Planta No Sind.,TC. Vendedor,BOUTIQUES,Ventas Sfera Gran Plaza,Operadora Sfera México SA,361,0
117153,15099771,1998-05-26,1998,5,26,Hombre,0,2019-09-02,2019,9,21,2020-01-28,2020,1,0,21,2020,Baja,3.00,Sin Dato,5109,Sfera Galerias Lag Torreon,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Ventas Sfera BT Gal Lagun,Operadora Sfera México SA,148,0


#Drop de columnas que no vamos a utilizar

In [ ]:
columns_to_drop = ['Ultima Evaluación', 'CP Vivienda', 'NDia', 'IAño', 'CP Trabajo', 'Fecha nacimiento', 'Fecha ingreso', 'Antigüedad', 'Fecha Salida', 'Unidad', 'Año salida', 'Desc Medida', 'Genero', 'SAño', 'Edad salida','SMes', 'Duración Días']
Dataset_2 = Demo.drop(columns=columns_to_drop)
Dataset_2

,Nº pers.,NAño,NMes,No Hijos,IMes,Edad ingreso,Locación,Desc Fun,GroPer,Área Personal,Departamento,Desc Soc,Antigüedad_Grupos
19,13307867,1997,3,0,6,18,Suburbia Los Cabos Patio,Cajero,SBB Planta,SBB Operación,CAJAS,Suburbia S. de R.L. de CV,2
20,14877876,1991,5,0,2,27,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,"Distribuidora Liverpool,",1
21,15228049,1992,3,2,12,27,Boutique Los Cabos,Consejero de Belleza,Vía Planta No Sind.,Personal General,FRAGANCIAS,"Distribuidora Liverpool,",1
22,12379884,1970,3,0,9,42,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,Operadora Comercial Liver,2
23,14559473,1965,1,0,5,53,Boutique Los Cabos,Consejero de Belleza,Planta No Sind.,Personal General,FRAGANCIAS,"Distribuidora Liverpool,",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
117144,14503279,1996,7,0,4,21,Sfera Tlaquepaque,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Operadora Sfera México SA,1
117145,14583940,1996,1,0,6,22,Sfera Playa del Carmen,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Operadora Sfera México SA,0
117149,14864493,1990,8,0,1,28,Sfera Guadalajara Gran Plaza,Vendedor Boutique Sfera,Planta No Sind.,TC. Vendedor,BOUTIQUES,Operadora Sfera México SA,0
117153,15099771,1998,5,0,9,21,Sfera Galerias Lag Torreon,Vendedor Boutique Sfera,Planta No Sind.,MT. Vendedor,BOUTIQUES,Operadora Sfera México SA,0


In [ ]:
# Change columns names so they are easier to understand
column_name_mapping = {'NAño': 'Año Nacimiento', 'NMes': 'Mes Nacimiento', 'IMes': 'Mes Ingreso', 'GroPer': 'Gpo Personal', 'Desc Fun': 'Función', 'Locación': 'Ubicación'}

# Use the rename method to change column names
Dataset_2 = Dataset_2.rename(columns=column_name_mapping)

In [ ]:
# Guardar el DataFrame Dataset_2 en un nuevo archivo CSV
demo_limpiado_path = '/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/DatasetModelado.csv'
Dataset_2.to_csv(demo_limpiado_path, index=False)

# ***2.***

#Carga de Dataset Modelado

In [ ]:
drive.mount('/content/drive')
path='/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/'
Dataset_2=pd.read_csv(path+'DatasetModelado.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
display(Dataset_2.head(n=2))

,Año Nacimiento,Mes Nacimiento,No Hijos,Mes Ingreso,Edad ingreso,Ubicación,Función,Gpo Personal,Área Personal,Departamento,Desc Soc,Antigüedad_Grupos
0,1997,3,0,6,18,Suburbia Los Cabos Patio,Cajero,SBB Planta,SBB Operación,CAJAS,Suburbia S. de R.L. de CV,2
1,1991,5,0,2,27,Boutique Los Cabos,Vendedor Bilingue,Planta No Sind.,TC. Vendedor,FRAGANCIAS,"Distribuidora Liverpool,",1


#Target Encoding (''Antigüedad en Grupos'')

In [ ]:
#target encoding para la columna 'GroPer'
encoder1 = TargetEncoder(cols = ['Gpo Personal'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Gpo_Personal_TE'] = encoder1.fit_transform(Dataset_2['Gpo Personal'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Gpo Personal', axis=1)

In [ ]:
#target encoding para la columna 'Departamento'
encoder2 = TargetEncoder(cols = ['Departamento'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Departamento_TE'] = encoder2.fit_transform(Dataset_2['Departamento'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Departamento', axis=1)

In [ ]:
#target encoding para la columna 'Desc Fun'
encoder3 = TargetEncoder(cols = ['Función'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Función_TE'] = encoder3.fit_transform(Dataset_2['Función'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Función', axis=1)

In [ ]:
#target encoding para la columna 'Locación'
encoder4 = TargetEncoder(cols = ['Ubicación'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Ubicación_TE'] = encoder4.fit_transform(Dataset_2['Ubicación'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Ubicación', axis=1)

In [ ]:
#target encoding para la columna 'Locación'
encoder5 = TargetEncoder(cols = ['Área Personal'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Área_Personal_TE'] = encoder5.fit_transform(Dataset_2['Área Personal'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Área Personal', axis=1)

In [ ]:
#target encoding para la columna 'Locación'
encoder6 = TargetEncoder(cols = ['Desc Soc'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Desc_Soc_TE'] = encoder6.fit_transform(Dataset_2['Desc Soc'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Desc Soc', axis=1)

In [ ]:
 # Eliminamos todas las filas con valores Nan
Dataset_2 = Dataset_2.dropna()
Dataset_2 = Dataset_2.reset_index(drop=True)
Dataset_2

,Año Nacimiento,Mes Nacimiento,No Hijos,Mes Ingreso,Edad ingreso,Antigüedad_Grupos,Gpo_Personal_TE,Departamento_TE,Función_TE,Ubicación_TE,Área_Personal_TE,Desc_Soc_TE
0,1997,3,0,6,18,2,0.703896,0.765256,0.706913,0.847862,0.696924,0.732517
1,1991,5,0,2,27,1,0.988776,1.153992,1.000000,0.847862,0.979834,0.921090
2,1992,3,2,12,27,1,0.605447,1.153992,1.170040,0.847862,0.758548,0.921090
3,1970,3,0,9,42,2,0.988776,1.153992,1.000000,0.847862,0.979834,0.877903
4,1965,1,0,5,53,1,0.988776,1.153992,1.170040,0.847862,0.758548,0.921090
...,...,...,...,...,...,...,...,...,...,...,...,...
61099,1996,7,0,4,21,1,0.988776,0.903794,0.811570,0.847862,0.841567,0.789346
61100,1996,1,0,6,22,0,0.988776,0.903794,0.811570,0.750000,0.841567,0.789346
61101,1990,8,0,1,28,0,0.988776,0.903794,0.811570,0.847862,0.979834,0.789346
61102,1998,5,0,9,21,0,0.988776,0.903794,0.811570,0.847862,0.841567,0.789346


#Entrenamiento del Modelo y Guardado

###Regresión Logística Multinomial ('Antigüedad en Grupos')

In [ ]:
X = Dataset_2.drop(columns=['Antigüedad_Grupos'], axis=1)
y = Dataset_2['Antigüedad_Grupos']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Creamos y entrenamos el modelo
clf_LR = LogisticRegression(multi_class='multinomial', solver='lbfgs')
clf_LR.fit(X_train_scaled, y_train)


LogisticRegression(multi_class='multinomial')

In [ ]:
# Hacemos predicciones con el set de pruebas
y_pred = clf_LR.predict(X_test_scaled)


In [ ]:
# Evaluamos el modelo
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.78


In [ ]:
#Calculamos la precision y recall
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# Imprimimos resultados
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

Precision: 0.7753
Recall: 0.7772


In [ ]:
# Generamos e imprimimos un reporte de clasificacion
report = classification_report(y_test, y_pred)

print("Classification Report:\n", report)

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.59      0.65      3622
           1       0.77      0.85      0.81      6791
           2       0.91      0.87      0.89      1808

    accuracy                           0.78     12221
   macro avg       0.80      0.77      0.78     12221
weighted avg       0.78      0.78      0.77     12221



###Random Forest ('Antigüedad en Grupos')

In [ ]:

X = Dataset_2.drop(columns=['Antigüedad_Grupos'], axis=1)
y = Dataset_2['Antigüedad_Grupos']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
clf_RF = RandomForestClassifier(n_estimators=70, random_state=42)  # You can adjust hyperparameters as needed
clf_RF.fit(X_train, y_train)

RandomForestClassifier(n_estimators=70, random_state=42)

In [ ]:
y_pred = clf_RF.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.78


In [ ]:
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

Precision: 0.7848
Recall: 0.7843


In [ ]:
# Generate a classification report
report = classification_report(y_test, y_pred)

# Print the classification report
print("Classification Report:\n", report)

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.61      0.66      3622
           1       0.77      0.87      0.82      6791
           2       0.92      0.82      0.87      1808

    accuracy                           0.78     12221
   macro avg       0.81      0.77      0.78     12221
weighted avg       0.78      0.78      0.78     12221



###Guardado de los Clasificadores, Scaler y Encoders

In [ ]:
#Guardamos el modelo en un archivo utilizando joblib
joblib.dump(clf_RF, 'RandomForest_clf.joblib')

['RandomForest_clf.joblib']

In [ ]:
joblib.dump(scaler, 'standard_scaler.joblib')

['standard_scaler.joblib']

In [ ]:
joblib.dump(clf_LR, 'LogisticRegression_clf.joblib')

['LogisticRegression_clf.joblib']

In [ ]:
joblib.dump(encoder1, 'target_encoder_Gpo_Personal.joblib')

['target_encoder_Gpo_Personal.joblib']

In [ ]:
joblib.dump(encoder2, 'target_encoder_Departamento.joblib')

['target_encoder_Departamento.joblib']

In [ ]:
joblib.dump(encoder3, 'target_encoder_Funcion.joblib')

['target_encoder_Funcion.joblib']

In [ ]:
joblib.dump(encoder4, 'target_encoder_Ubicacion.joblib')

['target_encoder_Ubicacion.joblib']

In [ ]:
joblib.dump(encoder5, 'target_encoder_Area_Personal.joblib')

['target_encoder_Area_Personal.joblib']

In [ ]:
joblib.dump(encoder6, 'target_encoder_Desc_Soc.joblib')

['target_encoder_Desc_Soc.joblib']

#Carga de Archivo Nuevo Proporcionado por Liverpool

In [ ]:
path='/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/'
DatasetLiverpool=pd.read_csv(path+'Ejemplo_Liverpool.csv')

In [ ]:
DatasetLiverpool

,Año Nacimiento,Mes Nacimiento,No Hijos,Mes Ingreso,Edad ingreso,Gpo Personal,Departamento,Función,Ubicación,Área Personal,Desc Soc
0,1991,5,0,2,27,Planta No Sind.,FRAGANCIAS,Vendedor Bilingue,Boutique Los Cabos,TC. Vendedor,"Distribuidora Liverpool,"


#Target Encoding Columnas Categóricas

In [ ]:
# Cargamos el encoder
target_encoder_Gpo_Personal = joblib.load('target_encoder_Gpo_Personal.joblib')

#Preparamos el encoder con nuestro dataset
DatasetLiverpool['Gpo_Personal_TE'] = target_encoder_Gpo_Personal.transform(DatasetLiverpool['Gpo Personal'])

# Eliminamos la columna original
DatasetLiverpool = DatasetLiverpool.drop('Gpo Personal', axis=1)

In [ ]:
# Cargamos el encoder
target_encoder_Departamento = joblib.load('target_encoder_Departamento.joblib')

#Preparamos el encoder con nuestro dataset
DatasetLiverpool['Departamento_TE'] = target_encoder_Departamento.transform(DatasetLiverpool['Departamento'])

# Eliminamos la columna original
DatasetLiverpool = DatasetLiverpool.drop('Departamento', axis=1)

In [ ]:
# Cargamos el encoder
target_encoder_Funcion = joblib.load('target_encoder_Funcion.joblib')

#Preparamos el encoder con nuestro dataset
DatasetLiverpool['Función_TE'] = target_encoder_Funcion.transform(DatasetLiverpool['Función'])

# Eliminamos la columna original
DatasetLiverpool = DatasetLiverpool.drop('Función', axis=1)

In [ ]:
# Cargamos el encoder
target_encoder_Ubicacion = joblib.load('target_encoder_Ubicacion.joblib')

#Preparamos el encoder con nuestro dataset
DatasetLiverpool['Ubicación_TE'] = target_encoder_Ubicacion.transform(DatasetLiverpool['Ubicación'])

# Eliminamos la columna original
DatasetLiverpool = DatasetLiverpool.drop('Ubicación', axis=1)

In [ ]:
# Cargamos el encoder
target_encoder_Area_Personal = joblib.load('target_encoder_Area_Personal.joblib')

#Preparamos el encoder con nuestro dataset
DatasetLiverpool['Área_Personal_TE'] = target_encoder_Area_Personal.transform(DatasetLiverpool['Área Personal'])

# Eliminamos la columna original
DatasetLiverpool = DatasetLiverpool.drop('Área Personal', axis=1)

In [ ]:
# Cargamos el encoder
target_encoder_Desc_Soc = joblib.load('target_encoder_Desc_Soc.joblib')

#Preparamos el encoder con nuestro dataset
DatasetLiverpool['Desc_Soc_TE'] = target_encoder_Desc_Soc.transform(DatasetLiverpool['Desc Soc'])

# Eliminamos la columna original
DatasetLiverpool = DatasetLiverpool.drop('Desc Soc', axis=1)

In [ ]:
DatasetLiverpool

,Año Nacimiento,Mes Nacimiento,No Hijos,Mes Ingreso,Edad ingreso,Gpo_Personal_TE,Departamento_TE,Función_TE,Ubicación_TE,Área_Personal_TE,Desc_Soc_TE
0,1991,5,0,2,27,0.988776,1.153992,1.0,0.847862,0.979834,0.92109


#Predicción del Modelo

##Regresión Logística Multinomial

In [ ]:
# Cargamos el scaler y transformamos el nuevo set de datos
standard_scaler = joblib.load('standard_scaler.joblib')

scaled_data = standard_scaler.transform(DatasetLiverpool)

In [ ]:
LogisticRegression_clf = joblib.load('LogisticRegression_clf.joblib')

In [ ]:
#Dividimos en 3 clases
class_mapping = {0: 'en menos de 1 año', 1: 'entre 1 y 5 años', 2: 'en más de 5 años'}

# Predecimos con los mismos datos
predictions_LR_numeric = LogisticRegression_clf.predict(scaled_data)

# Convertimos a string las predicciones
predictions_LR_string = [class_mapping[pred] for pred in predictions_LR_numeric]
predictions_LR_string

['entre 1 y 5 años']

In [ ]:
#Asume que contamos con las probabilidades de prediccion para cada clase
predicted_probabilities_LR = LogisticRegression_clf.predict_proba(scaled_data)

# Definimos nombres de clases
class_labels = ['Menos de un Año', 'Entre 1 y 5 Años', 'Más de 5 Años']

#Convertimos cada linea de probabilidades en lista de strings

class_prob_strings_LR = []
for probs in predicted_probabilities_LR:
    class_probs = [f"{class_labels[i]}: {prob * 100:.2f}%" for i, prob in enumerate(probs)]

    class_prob_strings_LR.append(class_probs)

#Cada string dentro de class_prob_strings_LR representa la probabilidad de cada clase en especifico
for class_probs in class_prob_strings_LR:
    print(class_probs)

['Menos de un Año: 19.14%', 'Entre 1 y 5 Años: 80.61%', 'Más de 5 Años: 0.25%']


##Random Forest

In [ ]:
RandomForest_clf = joblib.load('RandomForest_clf.joblib')

In [ ]:
# Definimos cada string con su respectiva clase
class_mapping = {0: 'en menos de 1 año', 1: 'entre 1 y 5 años', 2: 'en más de 5 años'}

# Hacemos predicciones con el nuevo dataset
predictions_RF_numeric = RandomForest_clf.predict(DatasetLiverpool)

# Mapeamos predicciones numericas en strings
predictions_RF_string = [class_mapping[pred] for pred in predictions_RF_numeric]
predictions_RF_string

['entre 1 y 5 años']

In [ ]:
# Asumiendi que tenemos las probabilidades predecidas por cada clase
predicted_probabilities_RF = RandomForest_clf.predict_proba(DatasetLiverpool)

# Definimos los nombres de las clases
class_labels = ['Menos de un Año', 'Entre 1 y 5 Años', 'Más de 5 Años']

#Convertimos cada fila de probabilidades en listas de strings
class_prob_strings_RF = []
for probs in predicted_probabilities_RF:
    class_probs = [f"{class_labels[i]}: {prob * 100:.2f}%" for i, prob in enumerate(probs)]

    class_prob_strings_RF.append(class_probs)

#Cada string dentro de class_prob_strings_RF representa la probabilidad de cada clase en especifico
for class_probs in class_prob_strings_RF:
    print(class_probs)

['Menos de un Año: 7.14%', 'Entre 1 y 5 Años: 91.43%', 'Más de 5 Años: 1.43%']
